# LeetCode 1618 — Maximum Font to Fit a Sentence in a Screen bằng Python

Link đề: https://leetcode.com/problems/maximum-font-to-fit-a-sentence-in-a-screen/ · Độ khó: **Medium** · Chủ đề: Array, String, Binary Search, Interactive

> **Ghi chú (bài Premium 🔒):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc. Đây là bài **interactive**: lớp `FontInfo` thật bị LeetCode giấu đi; notebook tự dựng một `FontInfo` **giả lập** thỏa đúng các tính chất đơn điệu của đề và cho ra đúng output của 3 ví dụ.

Skeleton LeetCode:

```python
# """
# This is FontInfo's API interface.
# You should not implement it, or speculate about its implementation
# """
# class FontInfo(object):
#    Return the width of char ch when fontSize is used.
#    def getWidth(self, fontSize, ch):
#        """
#        :type fontSize: int
#        :type ch: char
#        :rtype int
#        """
#
#    def getHeight(self, fontSize):
#        """
#        :type fontSize: int
#        :rtype int
#        """
class Solution:
    def maxFont(self, text: str, w: int, h: int, fonts: List[int], fontInfo: 'FontInfo') -> int:
        ...
```

## 1. Mục tiêu học

1. Làm quen với bài **interactive**: chỉ được hỏi thông tin qua một **API** (`getWidth`, `getHeight`), và học cách tự dựng API giả lập để thử nghiệm.
2. Nhận ra **tính đơn điệu**: font càng lớn càng khó vừa ⇒ dãy "vừa / không vừa" theo chỉ số font có dạng `T T … T F F … F`.
3. Thành thạo **tìm kiếm nhị phân "phần tử True cuối cùng"** và tránh các lỗi kinh điển (vòng lặp vô hạn, lệch 1, quên trường hợp không có đáp án).
4. Tối ưu số lần gọi API bằng **đếm tần suất ký tự** (`Counter`): tính chiều rộng bằng ≤ 26 lần gọi thay vì `len(text)` lần.
5. Đo và so sánh số lần gọi API giữa các cách: vét cạn, quét có dừng sớm, nhị phân, nhị phân + đếm tần suất.

## 2. Đề bài & ví dụ

Cho chuỗi `text` cần hiển thị trên **một dòng** của màn hình rộng `w`, cao `h`. Ta được chọn một cỡ chữ trong mảng `fonts` (đã sắp **tăng dần**, không trùng).

Interface `FontInfo` cho biết:
- `getWidth(fontSize, ch)`: chiều rộng ký tự `ch` ở cỡ `fontSize` — `O(1)` mỗi lần gọi;
- `getHeight(fontSize)`: chiều cao của mọi ký tự ở cỡ `fontSize` — `O(1)` mỗi lần gọi.

**Chiều rộng** của `text` ở cỡ `fs` là **tổng** `getWidth(fs, text[i])` với mọi `i`. **Chiều cao** là `getHeight(fs)`. `text` **vừa** màn hình ở cỡ `fs` nếu chiều rộng `≤ w` **và** chiều cao `≤ h`.

Đề **đảm bảo**: gọi lại với cùng tham số trả cùng kết quả, và với mọi `fs`, `ch`:
- `getHeight(fs) ≤ getHeight(fs + 1)`;
- `getWidth(fs, ch) ≤ getWidth(fs + 1, ch)`.

Trả về **cỡ chữ lớn nhất** trong `fonts` làm `text` vừa màn hình; nếu không cỡ nào vừa, trả về `-1`.

**Ví dụ 1:** `text = "helloworld", w = 80, h = 20, fonts = [6,8,10,12,14,16,18,24,36]` → `6`.

**Ví dụ 2:** `text = "leetcode", w = 1000, h = 50, fonts = [1,2,4]` → `4`.

**Ví dụ 3:** `text = "easyquestion", w = 100, h = 100, fonts = [10,15,20,25]` → `-1`.

**Ràng buộc:**
- `1 ≤ text.length ≤ 50000`, `text` chỉ gồm chữ thường.
- `1 ≤ w ≤ 10⁷`, `1 ≤ h ≤ 10⁴`.
- `1 ≤ fonts.length ≤ 10⁵`, `1 ≤ fonts[i] ≤ 10⁵`, `fonts` tăng dần, không trùng.

Đề **không** công bố `FontInfo` thật của các ví dụ. Bên dưới ta dựng một `FontInfo` giả lập (mục 3.1) cho ra đúng 3 output trên.

In [1]:
import random
import time
import bisect
from collections import Counter
from typing import List

class FontInfo(object):
    """FontInfo GIẢ LẬP (không phải của LeetCode).
    - getWidth(fs, ch) = fs + (1 nếu ch thuộc 'n'..'z', ngược lại 0)  -> không giảm theo fs
    - getHeight(fs)    = fs                                          -> không giảm theo fs
    Có đếm số lần gọi để so sánh các thuật toán.
    """
    def __init__(self):
        self.calls_w = 0
        self.calls_h = 0
    def getWidth(self, fontSize, ch):
        self.calls_w += 1
        return fontSize + (ord(ch) - ord("a")) // 13
    def getHeight(self, fontSize):
        self.calls_h += 1
        return fontSize
    def reset(self):
        self.calls_w = self.calls_h = 0

examples = [
    ("helloworld", 80, 20, [6, 8, 10, 12, 14, 16, 18, 24, 36], 6),
    ("leetcode", 1000, 50, [1, 2, 4], 4),
    ("easyquestion", 100, 100, [10, 15, 20, 25], -1),
]
fi = FontInfo()
for text, w, h, fonts, exp in examples:
    widths = {fs: sum(fi.getWidth(fs, c) for c in text) for fs in fonts}
    print(f"{text!r}: w={w}, h={h}, chiều rộng theo cỡ = {widths} -> kỳ vọng {exp}")

'helloworld': w=80, h=20, chiều rộng theo cỡ = {6: 64, 8: 84, 10: 104, 12: 124, 14: 144, 16: 164, 18: 184, 24: 244, 36: 364} -> kỳ vọng 6
'leetcode': w=1000, h=50, chiều rộng theo cỡ = {1: 10, 2: 18, 4: 34} -> kỳ vọng 4
'easyquestion': w=100, h=100, chiều rộng theo cỡ = {10: 128, 15: 188, 20: 248, 25: 308} -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Bài interactive và lớp giả lập

Trong bài interactive, LeetCode truyền vào một **đối tượng** (`fontInfo`) và ta chỉ được gọi phương thức của nó. Để tự test ở máy, ta viết một lớp cùng **tên phương thức, cùng tham số** (như `FontInfo` ở trên). Điều quan trọng là lớp giả lập phải thỏa **mọi giả thiết** của đề — ở đây là tính **không giảm** theo `fontSize`. Ta kiểm tra điều đó bằng code:

In [2]:
fi = FontInfo()
ok = all(fi.getWidth(fs, ch) <= fi.getWidth(fs + 1, ch)
         for fs in range(1, 300) for ch in "abcdefghijklmnopqrstuvwxyz")
ok = ok and all(fi.getHeight(fs) <= fi.getHeight(fs + 1) for fs in range(1, 300))
print("FontInfo giả lập đơn điệu:", ok, "| ví dụ getWidth(10,'a') =", fi.getWidth(10, "a"), ", getWidth(10,'z') =", fi.getWidth(10, "z"))

FontInfo giả lập đơn điệu: True | ví dụ getWidth(10,'a') = 10 , getWidth(10,'z') = 11


### 3.2 Tổng bằng generator và `Counter`

`sum(f(c) for c in text)` gọi `f` đúng `len(text)` lần. Nếu `f` chỉ phụ thuộc vào **ký tự** (không phụ thuộc vị trí), ta gom các ký tự giống nhau: `Counter(text)` cho `{ký tự: số lần}` và tính `sum(f(c) * k for c, k in cnt.items())` — tối đa 26 lần gọi.

In [3]:
text = "helloworld"
cnt = Counter(text)
print(cnt)
fi = FontInfo()
s1 = sum(fi.getWidth(6, c) for c in text); calls1 = fi.calls_w; fi.reset()
s2 = sum(fi.getWidth(6, c) * k for c, k in cnt.items()); calls2 = fi.calls_w
print("tổng trực tiếp:", s1, "(", calls1, "lần gọi) | tổng qua Counter:", s2, "(", calls2, "lần gọi)")

Counter({'l': 3, 'o': 2, 'h': 1, 'e': 1, 'w': 1, 'r': 1, 'd': 1})
tổng trực tiếp: 64 ( 10 lần gọi) | tổng qua Counter: 64 ( 7 lần gọi)


### 3.3 Tìm kiếm nhị phân "phần tử True cuối cùng"

Cho hàm `ok(i)` **đơn điệu giảm**: `ok(0..k) = True`, `ok(k+1..) = False`. Cần tìm `k` (hoặc `−1` nếu không có True). Mẫu an toàn nhất:

```python
lo, hi, ans = 0, n - 1, -1
while lo <= hi:
    mid = (lo + hi) // 2
    if ok(mid): ans = mid; lo = mid + 1   # mid đạt, thử lớn hơn
    else:       hi = mid - 1             # mid hỏng, mọi chỉ số lớn hơn cũng hỏng
return ans
```

In [4]:
def last_true(n, ok):
    lo, hi, ans = 0, n - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if ok(mid):
            ans, lo = mid, mid + 1
        else:
            hi = mid - 1
    return ans

arr = [True, True, True, False, False]
print(last_true(len(arr), lambda i: arr[i]), last_true(3, lambda i: False), last_true(4, lambda i: True))

2 -1 3


### 3.4 `bisect` với tham số `key` (Python ≥ 3.10)

`bisect.bisect_left(a, x, key=f)` tìm vị trí chèn `x` vào dãy `f(a[i])` (đã sắp tăng). Nếu `f(i) = 0` khi vừa và `1` khi không vừa (dãy `0 0 0 1 1` tăng dần), thì `bisect_left(range(n), 1, key=f)` là chỉ số **đầu tiên không vừa** ⇒ đáp án là phần tử ngay trước đó.

In [5]:
bad = [0, 0, 0, 1, 1]
first_bad = bisect.bisect_left(range(len(bad)), 1, key=lambda i: bad[i])
print("chỉ số đầu tiên hỏng:", first_bad, "-> True cuối cùng:", first_bad - 1)

chỉ số đầu tiên hỏng: 3 -> True cuối cùng: 2


## 4. Ý tưởng nền tảng / trực giác

Tính bằng tay **Ví dụ 1** với `FontInfo` giả lập: `text = "helloworld"` (10 ký tự, trong đó `o, w, o, r` thuộc `n..z` ⇒ mỗi ký tự rộng thêm 1).

| cỡ | chiều rộng = 10·cỡ + 4 | ≤ 80? | chiều cao = cỡ | ≤ 20? | vừa? |
|---|---|---|---|---|---|
| 6 | 64 | có | 6 | có | **có** |
| 8 | 84 | không | 8 | có | không |
| 10 | 104 | không | 10 | có | không |
| … | tăng dần | không | | | không |

Ngay khi cỡ 8 không vừa, mọi cỡ lớn hơn **chắc chắn** không vừa (chiều rộng từng ký tự không giảm). Vậy dãy "vừa?" theo chỉ số luôn có dạng **`có có … có không … không`** — đây là tín hiệu cho **tìm kiếm nhị phân**.

In [6]:
def fits(text, w, h, fs, fontInfo):
    if fontInfo.getHeight(fs) > h:
        return False
    return sum(fontInfo.getWidth(fs, c) for c in text) <= w

fi = FontInfo()
for text, w, h, fonts, exp in examples:
    print(f"{text!r}:", ["có" if fits(text, w, h, fs, fi) else "không" for fs in fonts])

'helloworld': ['có', 'không', 'không', 'không', 'không', 'không', 'không', 'không', 'không']
'leetcode': ['có', 'có', 'có']
'easyquestion': ['không', 'không', 'không', 'không']


## 5. Cách 1 — Brute Force / baseline: thử mọi cỡ chữ

**Ý tưởng:** duyệt **mọi** cỡ trong `fonts`, cỡ nào vừa thì cập nhật đáp án (vì `fonts` tăng dần, cỡ vừa cuối cùng là lớn nhất).

In [7]:
def max_font_bruteforce(text, w, h, fonts, fontInfo):
    ans = -1
    for fs in fonts:
        if fontInfo.getHeight(fs) <= h and sum(fontInfo.getWidth(fs, c) for c in text) <= w:
            ans = fs
    return ans

for text, w, h, fonts, exp in examples:
    fi = FontInfo()
    got = max_font_bruteforce(text, w, h, fonts, fi)
    print(f"{got} kỳ vọng {exp} | getWidth gọi {fi.calls_w} lần, getHeight gọi {fi.calls_h} lần")
    assert got == exp

6 kỳ vọng 6 | getWidth gọi 70 lần, getHeight gọi 9 lần
4 kỳ vọng 4 | getWidth gọi 24 lần, getHeight gọi 3 lần
-1 kỳ vọng -1 | getWidth gọi 48 lần, getHeight gọi 4 lần


**Độ phức tạp:** `O(F · T)` với `F = len(fonts)`, `T = len(text)`. Tệ nhất `10⁵ · 5·10⁴ = 5·10⁹` lần gọi `getWidth` — **chắc chắn TLE**.

**Nút thắt:** (1) không tận dụng tính đơn điệu — vẫn thử cả những cỡ chắc chắn hỏng; (2) mỗi lần thử gọi `getWidth` `T` lần dù chỉ có ≤ 26 ký tự khác nhau.

Một cải tiến nhỏ: duyệt từ cỡ **lớn nhất xuống** và dừng ở cỡ vừa đầu tiên — tốt khi đáp án gần cuối, nhưng tệ nhất vẫn `O(F · T)` (khi đáp án là `-1` hoặc cỡ nhỏ nhất).

In [8]:
def max_font_desc(text, w, h, fonts, fontInfo):
    for fs in reversed(fonts):
        if fontInfo.getHeight(fs) <= h and sum(fontInfo.getWidth(fs, c) for c in text) <= w:
            return fs
    return -1

for text, w, h, fonts, exp in examples:
    fi = FontInfo(); got = max_font_desc(text, w, h, fonts, fi)
    print(f"{got} | getWidth gọi {fi.calls_w} lần")

6 | getWidth gọi 70 lần
4 | getWidth gọi 8 lần
-1 | getWidth gọi 48 lần


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đơn điệu:** đặt `ok(i)` = "cỡ `fonts[i]` vừa màn hình". Vì `fonts` tăng dần và `getWidth`, `getHeight` không giảm theo cỡ:
- Nếu `ok(i)` đúng **thì** `ok(j)` đúng với mọi `j < i` (cỡ nhỏ hơn: rộng và cao không lớn hơn).
- Nếu `ok(i)` sai **thì** `ok(j)` sai với mọi `j > i`.

⇒ Tìm **True cuối cùng** bằng nhị phân: chỉ `⌈log₂ F⌉ ≈ 17` lần kiểm tra thay vì `F = 10⁵`.

**Insight 2 — gom ký tự:** chiều rộng chỉ phụ thuộc vào **tần suất** ký tự: `width(fs) = Σ_c cnt[c] · getWidth(fs, c)` ⇒ mỗi lần kiểm tra chỉ ≤ 26 lần gọi `getWidth`.

**Insight 3 — kiểm tra chiều cao trước:** `getHeight` chỉ 1 lần gọi; nếu cao quá thì bỏ qua luôn phần tính chiều rộng.

**Invariant nhị phân:** mọi chỉ số `< lo` đều đã biết là vừa (hoặc `lo = 0`), mọi chỉ số `> hi` đều đã biết là không vừa; `ans` là chỉ số vừa lớn nhất đã thấy.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quét tuyến tính + đếm tần suất

**Giải thích từng đoạn:**
1. `cnt = Counter(text)` — một lần, `O(T)`.
2. Duyệt cỡ từ **lớn xuống nhỏ**; với mỗi cỡ: kiểm tra chiều cao, rồi tính chiều rộng qua `cnt` (≤ 26 lần gọi).
3. Cỡ vừa đầu tiên gặp được là đáp án.

Tệ nhất `O(T + 26·F) ≈ 2,6·10⁶` lần gọi — đã đủ qua giới hạn, nhưng còn tốt hơn được.

In [9]:
def max_font_scan_count(text, w, h, fonts, fontInfo):
    cnt = Counter(text)
    for fs in reversed(fonts):
        if fontInfo.getHeight(fs) > h:
            continue
        if sum(fontInfo.getWidth(fs, c) * k for c, k in cnt.items()) <= w:
            return fs
    return -1

for text, w, h, fonts, exp in examples:
    fi = FontInfo(); got = max_font_scan_count(text, w, h, fonts, fi)
    print(f"{got} kỳ vọng {exp} | getWidth gọi {fi.calls_w} lần"); assert got == exp

6 kỳ vọng 6 | getWidth gọi 49 lần
4 kỳ vọng 4 | getWidth gọi 6 lần
-1 kỳ vọng -1 | getWidth gọi 40 lần


**Code nộp LeetCode (Cách 2):**

In [10]:
class SolutionScan:
    def maxFont(self, text: str, w: int, h: int, fonts: List[int], fontInfo: 'FontInfo') -> int:
        cnt = Counter(text)
        for fs in reversed(fonts):
            if fontInfo.getHeight(fs) > h:
                continue
            if sum(fontInfo.getWidth(fs, c) * k for c, k in cnt.items()) <= w:
                return fs
        return -1

print([SolutionScan().maxFont(t, w, h, f, FontInfo()) for t, w, h, f, _ in examples])

[6, 4, -1]


**Độ phức tạp Cách 2:** Time `O(T + Σ·F)` với `Σ ≤ 26` = số ký tự khác nhau; Space `O(Σ)`.

### 7.2 Cách 3 — Tìm kiếm nhị phân, kiểm tra bằng cách cộng trực tiếp

**Giải thích từng đoạn:**
1. `fits(fs)`: chiều cao trước, rồi `sum(getWidth(fs, c) for c in text) ≤ w`.
2. Nhị phân trên chỉ số `0..F−1` theo mẫu "True cuối cùng" (mục 3.3).
3. Trả `fonts[ans]` nếu `ans ≠ −1`, ngược lại `−1`.

In [11]:
def max_font_binary(text, w, h, fonts, fontInfo):
    def fits(fs):
        if fontInfo.getHeight(fs) > h:
            return False
        return sum(fontInfo.getWidth(fs, c) for c in text) <= w
    lo, hi, ans = 0, len(fonts) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if fits(fonts[mid]):
            ans, lo = mid, mid + 1
        else:
            hi = mid - 1
    return fonts[ans] if ans != -1 else -1

for text, w, h, fonts, exp in examples:
    fi = FontInfo(); got = max_font_binary(text, w, h, fonts, fi)
    print(f"{got} kỳ vọng {exp} | getWidth gọi {fi.calls_w} lần"); assert got == exp

6 kỳ vọng 6 | getWidth gọi 30 lần
4 kỳ vọng 4 | getWidth gọi 16 lần
-1 kỳ vọng -1 | getWidth gọi 24 lần


**Code nộp LeetCode (Cách 3):**

In [12]:
class SolutionBinary:
    def maxFont(self, text: str, w: int, h: int, fonts: List[int], fontInfo: 'FontInfo') -> int:
        def fits(fs: int) -> bool:
            if fontInfo.getHeight(fs) > h:
                return False
            return sum(fontInfo.getWidth(fs, c) for c in text) <= w

        lo, hi, ans = 0, len(fonts) - 1, -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if fits(fonts[mid]):
                ans, lo = fonts[mid], mid + 1
            else:
                hi = mid - 1
        return ans

print([SolutionBinary().maxFont(t, w, h, f, FontInfo()) for t, w, h, f, _ in examples])

[6, 4, -1]


**Độ phức tạp Cách 3:** Time `O(T · log F)` ≈ `5·10⁴ · 17 ≈ 8,5·10⁵` lần gọi; Space `O(1)`.

### 7.3 Cách 4 — Tìm kiếm nhị phân + đếm tần suất (tối ưu)

Kết hợp cả hai insight: `Counter(text)` một lần, mỗi lần kiểm tra chỉ ≤ 26 lần gọi `getWidth` ⇒ tổng `O(T + Σ · log F)`.

In [13]:
def max_font_binary_count(text, w, h, fonts, fontInfo):
    cnt = Counter(text)
    def fits(fs):
        if fontInfo.getHeight(fs) > h:
            return False
        return sum(fontInfo.getWidth(fs, c) * k for c, k in cnt.items()) <= w
    lo, hi, ans = 0, len(fonts) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if fits(fonts[mid]):
            ans, lo = fonts[mid], mid + 1
        else:
            hi = mid - 1
    return ans

for text, w, h, fonts, exp in examples:
    fi = FontInfo(); got = max_font_binary_count(text, w, h, fonts, fi)
    print(f"{got} kỳ vọng {exp} | getWidth gọi {fi.calls_w} lần, getHeight gọi {fi.calls_h} lần"); assert got == exp

6 kỳ vọng 6 | getWidth gọi 21 lần, getHeight gọi 3 lần
4 kỳ vọng 4 | getWidth gọi 12 lần, getHeight gọi 2 lần
-1 kỳ vọng -1 | getWidth gọi 20 lần, getHeight gọi 2 lần


**Code nộp LeetCode (Cách 4):** xem mục 9 (bản được chọn để nộp).

**Độ phức tạp Cách 4:** Time `O(T + Σ · log F)`, Space `O(Σ)`.

### 7.4 Cách 5 — Viết gọn bằng `bisect` với `key`

Dãy `0 (vừa) … 0, 1 (không vừa) … 1` tăng dần theo chỉ số, nên `bisect_left(..., 1, key=...)` trả chỉ số **đầu tiên không vừa** `p`. Đáp án là `fonts[p − 1]` nếu `p > 0`, ngược lại `−1`.

In [14]:
class SolutionBisect:
    def maxFont(self, text: str, w: int, h: int, fonts: List[int], fontInfo: 'FontInfo') -> int:
        cnt = Counter(text)
        def bad(fs: int) -> int:
            if fontInfo.getHeight(fs) > h:
                return 1
            return int(sum(fontInfo.getWidth(fs, c) * k for c, k in cnt.items()) > w)
        p = bisect.bisect_left(fonts, 1, key=bad)
        return fonts[p - 1] if p > 0 else -1

print([SolutionBisect().maxFont(t, w, h, f, FontInfo()) for t, w, h, f, _ in examples])

[6, 4, -1]


Lưu ý: `key` của `bisect` có từ **Python 3.10**. Nếu môi trường cũ hơn, dùng vòng `while` của Cách 4.

**Độ phức tạp Cách 5:** giống Cách 4.

## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 1** (`fonts = [6,8,10,12,14,16,18,24,36]`, `w = 80`, `h = 20`):

| vòng | lo | hi | mid | cỡ | cao | rộng | vừa? | ans |
|---|---|---|---|---|---|---|---|---|
| 1 | 0 | 8 | 4 | 14 | 14 | 144 | không | −1 |
| 2 | 0 | 3 | 1 | 8 | 8 | 84 | không | −1 |
| 3 | 0 | 0 | 0 | 6 | 6 | 64 | có | 6 |
| kết thúc | 1 | 0 | | | | | | **6** |

Chỉ 3 lần kiểm tra thay vì 9. Hàm `trace_binary`:

In [15]:
def trace_binary(text, w, h, fonts, fontInfo):
    cnt = Counter(text)
    lo, hi, ans, step = 0, len(fonts) - 1, -1, 0
    print("cnt =", dict(cnt))
    while lo <= hi:
        step += 1
        mid = (lo + hi) // 2
        fs = fonts[mid]
        H = fontInfo.getHeight(fs)
        if H > h:
            W, ok = None, False
        else:
            W = sum(fontInfo.getWidth(fs, c) * k for c, k in cnt.items())
            ok = W <= w
        if ok:
            ans, lo = fs, mid + 1
        else:
            hi = mid - 1
        print(f"vòng {step}: mid={mid}, cỡ={fs:>3}, cao={H:>3}{'>h' if H > h else '  '}, rộng={W}, vừa={ok!s:<5} -> lo={lo}, hi={hi}, ans={ans}")
    print("Kết quả:", ans, "| getWidth gọi", fontInfo.calls_w, "lần")
    return ans

assert trace_binary(*examples[0][:4], FontInfo()) == 6

cnt = {'h': 1, 'e': 1, 'l': 3, 'o': 2, 'w': 1, 'r': 1, 'd': 1}
vòng 1: mid=4, cỡ= 14, cao= 14  , rộng=144, vừa=False -> lo=0, hi=3, ans=-1
vòng 2: mid=1, cỡ=  8, cao=  8  , rộng=84, vừa=False -> lo=0, hi=0, ans=-1
vòng 3: mid=0, cỡ=  6, cao=  6  , rộng=64, vừa=True  -> lo=1, hi=0, ans=6
Kết quả: 6 | getWidth gọi 21 lần


Trace trên **Ví dụ 3** (không cỡ nào vừa ⇒ `hi` cứ giảm đến `−1`, `ans` giữ `−1`) và một ví dụ tự dựng mà **chiều cao** là yếu tố quyết định (`text = "abc"`, `w = 1000`, `h = 15`, `fonts = [5, 10, 15, 20, 25]` — ví dụ tự dựng với FontInfo giả lập, đã đối chiếu brute force; nên đối chiếu lại đề gốc):

In [16]:
assert trace_binary(*examples[2][:4], FontInfo()) == -1
print()
r = trace_binary("abc", 1000, 15, [5, 10, 15, 20, 25], FontInfo())
assert r == max_font_bruteforce("abc", 1000, 15, [5, 10, 15, 20, 25], FontInfo()) == 15

cnt = {'e': 2, 'a': 1, 's': 2, 'y': 1, 'q': 1, 'u': 1, 't': 1, 'i': 1, 'o': 1, 'n': 1}
vòng 1: mid=1, cỡ= 15, cao= 15  , rộng=188, vừa=False -> lo=0, hi=0, ans=-1
vòng 2: mid=0, cỡ= 10, cao= 10  , rộng=128, vừa=False -> lo=0, hi=-1, ans=-1
Kết quả: -1 | getWidth gọi 20 lần

cnt = {'a': 1, 'b': 1, 'c': 1}
vòng 1: mid=2, cỡ= 15, cao= 15  , rộng=45, vừa=True  -> lo=3, hi=4, ans=15
vòng 2: mid=3, cỡ= 20, cao= 20>h, rộng=None, vừa=False -> lo=3, hi=2, ans=15
Kết quả: 15 | getWidth gọi 3 lần


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — nhị phân + đếm tần suất. Tham số `fontInfo` do LeetCode truyền vào; **không** tự định nghĩa lại `FontInfo` khi nộp.

In [17]:
# """
# This is FontInfo's API interface.
# You should not implement it, or speculate about its implementation
# """
# class FontInfo(object):
#    def getWidth(self, fontSize, ch): ...
#    def getHeight(self, fontSize): ...
class Solution:
    def maxFont(self, text: str, w: int, h: int, fonts: List[int], fontInfo: 'FontInfo') -> int:
        cnt = Counter(text)

        def fits(fs: int) -> bool:
            if fontInfo.getHeight(fs) > h:
                return False
            total = 0
            for c, k in cnt.items():
                total += fontInfo.getWidth(fs, c) * k
                if total > w:              # dừng sớm khi đã vượt
                    return False
            return True

        lo, hi, ans = 0, len(fonts) - 1, -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if fits(fonts[mid]):
                ans = fonts[mid]
                lo = mid + 1
            else:
                hi = mid - 1
        return ans

for text, w, h, fonts, exp in examples:
    got = Solution().maxFont(text, w, h, fonts, FontInfo())
    print(got, exp); assert got == exp

6 6
4 4
-1 -1


## 10. Vì sao thuật toán đúng

**Bổ đề (đơn điệu).** Với `i < j` (tức `fonts[i] < fonts[j]`): áp dụng giả thiết `getWidth(fs, ch) ≤ getWidth(fs+1, ch)` liên tiếp từ `fonts[i]` tới `fonts[j]` ⇒ `getWidth(fonts[i], ch) ≤ getWidth(fonts[j], ch)` với mọi `ch`; cộng theo mọi ký tự của `text` ⇒ `width(fonts[i]) ≤ width(fonts[j])`. Tương tự `height(fonts[i]) ≤ height(fonts[j])`. Vậy `ok(j) ⇒ ok(i)`.

**Tính đúng của `fits` qua `Counter`.** `Σ_{i} getWidth(fs, text[i]) = Σ_c cnt[c] · getWidth(fs, c)` vì `getWidth` trả cùng giá trị cho cùng tham số (đề đảm bảo) — chỉ là nhóm các số hạng bằng nhau. Dừng sớm khi `total > w` an toàn vì mọi số hạng còn lại `≥ 0` (chiều rộng không âm).

**Tính đúng của nhị phân (invariant).** Gọi `k` = chỉ số vừa lớn nhất (hoặc `−1`). Luôn có: `k ∈ [lo − 1, hi]` và `ans = fonts[lo − 1]` nếu `lo > 0` (ngược lại `ans = −1`).
- `fits(mid)` đúng ⇒ `k ≥ mid` ⇒ đặt `lo = mid + 1`, `ans = fonts[mid]` giữ invariant.
- `fits(mid)` sai ⇒ `k < mid` (đơn điệu) ⇒ đặt `hi = mid − 1` giữ invariant.

Mỗi vòng `hi − lo` giảm ít nhất 1 ⇒ dừng khi `lo = hi + 1`, lúc đó `k ∈ [lo − 1, lo − 1]` ⇒ `k = lo − 1` và `ans` đúng bằng `fonts[k]` (hoặc `−1`). ∎

## 11. Phân tích Time / Space Complexity

Gọi `T = len(text) ≤ 5·10⁴`, `F = len(fonts) ≤ 10⁵`, `Σ` = số ký tự khác nhau (`≤ 26`). Chi phí chính là **số lần gọi API** (mỗi lần `O(1)`).

| Cách | Số lần gọi `getWidth` (tệ nhất) | Time | Space |
|---|---|---|---|
| 1. Thử mọi cỡ | `F · T ≈ 5·10⁹` | `O(F·T)` | `O(1)` |
| 1'. Từ lớn xuống, dừng sớm | `F · T` (khi đáp án nhỏ / `−1`) | `O(F·T)` | `O(1)` |
| 2. Quét + Counter | `Σ · F ≈ 2,6·10⁶` | `O(T + Σ·F)` | `O(Σ)` |
| 3. Nhị phân | `T · ⌈log₂F⌉ ≈ 8,5·10⁵` | `O(T log F)` | `O(1)` |
| 4/5. Nhị phân + Counter | `Σ · ⌈log₂F⌉ ≈ 442` | `O(T + Σ log F)` | `O(Σ)` |

Đo số lần gọi thực tế trên input lớn nhất (bỏ Cách 1 vì quá chậm):

In [18]:
rng = random.Random(2024)
T, F = 50000, 100000
big_text = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(T))
big_fonts = list(range(1, F + 1))
big_w, big_h = 10 ** 7, 10 ** 4
for name, f in [("Cách 2 (quét+Counter)", max_font_scan_count), ("Cách 3 (nhị phân)", max_font_binary),
                ("Cách 4 (nhị phân+Counter)", max_font_binary_count)]:
    fi = FontInfo(); t = time.perf_counter(); r = f(big_text, big_w, big_h, big_fonts, fi); dt = time.perf_counter() - t
    print(f"{name:<28} -> {r:>4} | getWidth: {fi.calls_w:>8} lần | getHeight: {fi.calls_h:>6} lần | {dt*1000:7.1f} ms")

Cách 2 (quét+Counter)        ->  199 | getWidth:   254852 lần | getHeight:  99802 lần |    33.1 ms
Cách 3 (nhị phân)            ->  199 | getWidth:   700000 lần | getHeight:     17 lần |    53.5 ms
Cách 4 (nhị phân+Counter)    ->  199 | getWidth:      364 lần | getHeight:     17 lần |     1.1 ms


## 12. Bộ test

Gồm: ví dụ đề, edge case (1 cỡ, 1 ký tự, vừa **khít** `= w`/`= h`, chỉ chiều cao chặn, không cỡ nào vừa, mọi cỡ đều vừa), và đối chiếu mọi cách trên nhiều `FontInfo` **ngẫu nhiên đơn điệu** (seed cố định) — để chắc chắn lời giải không "ăn may" nhờ một FontInfo cụ thể.

In [19]:
class RandomFontInfo(object):
    """FontInfo ngẫu nhiên nhưng đơn điệu: width(fs, ch) = (fs * p_ch) // q + c_ch, height(fs) = (fs * ph) // qh + ch0."""
    def __init__(self, rng):
        self.p = {ch: rng.randint(1, 5) for ch in "abcdefghijklmnopqrstuvwxyz"}
        self.c = {ch: rng.randint(0, 3) for ch in "abcdefghijklmnopqrstuvwxyz"}
        self.q = rng.randint(1, 4)
        self.ph, self.qh, self.h0 = rng.randint(1, 3), rng.randint(1, 3), rng.randint(0, 5)
        self.calls_w = self.calls_h = 0
    def getWidth(self, fontSize, ch):
        self.calls_w += 1
        return (fontSize * self.p[ch]) // self.q + self.c[ch]
    def getHeight(self, fontSize):
        self.calls_h += 1
        return (fontSize * self.ph) // self.qh + self.h0

ALL = {
    "brute": max_font_bruteforce, "desc": max_font_desc, "scan+cnt": max_font_scan_count,
    "binary": max_font_binary, "binary+cnt": max_font_binary_count,
    "SolScan": lambda *a: SolutionScan().maxFont(*a), "SolBinary": lambda *a: SolutionBinary().maxFont(*a),
    "SolBisect": lambda *a: SolutionBisect().maxFont(*a), "submit": lambda *a: Solution().maxFont(*a),
}
def run_all(text, w, h, fonts, make_fi):
    res = {k: f(text, w, h, fonts, make_fi()) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, res
    return next(iter(res.values()))

for text, w, h, fonts, exp in examples:
    assert run_all(text, w, h, fonts, FontInfo) == exp

edge = [
    ("a", 5, 5, [5], 5),                        # 1 cỡ, vừa khít cả w và h
    ("a", 4, 5, [5], -1),                       # rộng 5 > 4
    ("z", 6, 5, [5], 5),                        # 'z' rộng 6 = w (khít)
    ("helloworld", 64, 20, [6, 8], 6),          # rộng 64 = w (khít)
    ("abc", 1000, 15, [5, 10, 15, 20, 25], 15), # chỉ chiều cao chặn
    ("abc", 10 ** 7, 10 ** 4, [1, 2, 3], 3),    # mọi cỡ đều vừa
    ("abc", 1, 1, [1, 2, 3], -1),               # không cỡ nào vừa (rộng 3 > 1)
]
for text, w, h, fonts, exp in edge:
    assert max_font_bruteforce(text, w, h, fonts, FontInfo()) == exp, (text, w, h, fonts)
    assert run_all(text, w, h, fonts, FontInfo) == exp
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [20]:
rng = random.Random(1618)
dist = Counter()
for _ in range(600):
    seed = rng.randrange(10 ** 9)
    make_fi = lambda s=seed: RandomFontInfo(random.Random(s))       # mọi cách dùng CÙNG một FontInfo
    text = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(rng.randint(1, 30)))
    fonts = sorted(rng.sample(range(1, 200), rng.randint(1, 25)))
    w = rng.randint(1, 3000); h = rng.randint(1, 300)
    r = run_all(text, w, h, fonts, make_fi)
    dist["-1" if r == -1 else ("lớn nhất" if r == fonts[-1] else "ở giữa")] += 1
print("600 test ngẫu nhiên OK | phân bố đáp án:", dict(dist))

600 test ngẫu nhiên OK | phân bố đáp án: {'ở giữa': 388, '-1': 148, 'lớn nhất': 64}


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra chiều cao

In [21]:
def wrong_no_height(text, w, h, fonts, fontInfo):
    cnt = Counter(text)
    lo, hi, ans = 0, len(fonts) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if sum(fontInfo.getWidth(fonts[mid], c) * k for c, k in cnt.items()) <= w:   # SAI: thiếu chiều cao
            ans, lo = fonts[mid], mid + 1
        else:
            hi = mid - 1
    return ans

args = ("abc", 1000, 15, [5, 10, 15, 20, 25])
print("sai :", wrong_no_height(*args, FontInfo()), "| đúng:", Solution().maxFont(*args, FontInfo()))

sai : 25 | đúng: 15


**Vì sao sai:** chữ phải vừa **cả** chiều rộng **và** chiều cao. Cỡ 25 rộng đủ nhưng cao 25 > 15. **Sửa:** `if fontInfo.getHeight(fs) > h: return False` trước.

### Lỗi 2 — Nhị phân `lo = mid` với `mid = (lo + hi) // 2` ⇒ lặp vô hạn

Ta cài thêm "cầu chì" giới hạn số vòng để minh họa mà không treo notebook:

In [22]:
def wrong_infinite(text, w, h, fonts, fontInfo, guard=100):
    def fits(fs):
        return fontInfo.getHeight(fs) <= h and sum(fontInfo.getWidth(fs, c) for c in text) <= w
    lo, hi, loops = 0, len(fonts) - 1, 0
    while lo < hi:
        loops += 1
        if loops > guard:
            return f"KẸT: lặp quá {guard} vòng (lo={lo}, hi={hi})"
        mid = (lo + hi) // 2           # SAI khi đi cùng lo = mid: làm tròn xuống
        if fits(fonts[mid]):
            lo = mid
        else:
            hi = mid - 1
    return fonts[lo] if fits(fonts[lo]) else -1

print("sai :", wrong_infinite("leetcode", 1000, 50, [1, 2, 4], FontInfo()), "| đúng:", Solution().maxFont("leetcode", 1000, 50, [1, 2, 4], FontInfo()))

sai : KẸT: lặp quá 100 vòng (lo=1, hi=2) | đúng: 4


**Vì sao sai:** khi `hi = lo + 1`, `mid = lo`; nếu `fits(mid)` thì `lo = mid` không đổi ⇒ kẹt. **Sửa:** dùng `mid = (lo + hi + 1) // 2` (làm tròn lên) khi cập nhật `lo = mid`, hoặc dùng mẫu `while lo <= hi` với `lo = mid + 1` như mục 3.3.

### Lỗi 3 — Trả `fonts[lo]` mà không kiểm tra trường hợp không cỡ nào vừa

In [23]:
def wrong_no_minus1(text, w, h, fonts, fontInfo):
    def fits(fs):
        return fontInfo.getHeight(fs) <= h and sum(fontInfo.getWidth(fs, c) for c in text) <= w
    lo, hi = 0, len(fonts) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if fits(fonts[mid]):
            lo = mid
        else:
            hi = mid - 1
    return fonts[lo]                    # SAI: fonts[0] có thể cũng không vừa

t, w, h, f, exp = examples[2]
print("sai :", wrong_no_minus1(t, w, h, f, FontInfo()), "| đúng:", Solution().maxFont(t, w, h, f, FontInfo()))

sai : 10 | đúng: -1


**Vì sao sai:** vòng nhị phân dạng `lo < hi` luôn dừng ở **một** chỉ số, nhưng không đảm bảo chỉ số đó vừa. **Sửa:** `return fonts[lo] if fits(fonts[lo]) else -1`.

### Lỗi 4 — Trả về **chỉ số** thay vì **cỡ chữ**

In [24]:
def wrong_index(text, w, h, fonts, fontInfo):
    cnt = Counter(text)
    lo, hi, ans = 0, len(fonts) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if fontInfo.getHeight(fonts[mid]) <= h and sum(fontInfo.getWidth(fonts[mid], c) * k for c, k in cnt.items()) <= w:
            ans, lo = mid, mid + 1        # SAI: lưu chỉ số
        else:
            hi = mid - 1
    return ans

t, w, h, f, exp = examples[1]
print("sai :", wrong_index(t, w, h, f, FontInfo()), "| đúng:", Solution().maxFont(t, w, h, f, FontInfo()))

sai : 2 | đúng: 4


**Vì sao sai:** đề hỏi **cỡ chữ** (`fonts[i]`), không hỏi vị trí. **Sửa:** `ans = fonts[mid]`.

### Lỗi 5 — Dùng `<` thay vì `<=` (bỏ trường hợp vừa khít)

In [25]:
def wrong_strict(text, w, h, fonts, fontInfo):
    ans = -1
    for fs in fonts:
        if fontInfo.getHeight(fs) < h and sum(fontInfo.getWidth(fs, c) for c in text) < w:   # SAI
            ans = fs
    return ans

args = ("helloworld", 64, 20, [6, 8])     # rộng ở cỡ 6 đúng bằng 64 (ví dụ tự dựng với FontInfo giả lập)
print("sai :", wrong_strict(*args, FontInfo()), "| đúng:", Solution().maxFont(*args, FontInfo()))

sai : -1 | đúng: 6


**Vì sao sai:** "vừa màn hình" nghĩa là **không vượt quá**: `≤ w` và `≤ h`. **Sửa:** dùng `<=`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm giá trị **lớn nhất** thỏa điều kiện" + điều kiện **đơn điệu** (lớn hơn thì khó hơn) ⇒ **binary search on answer**.
- Mảng ứng viên đã **sắp tăng** sẵn (`fonts`) ⇒ nhị phân trên **chỉ số**.
- Tổng `Σ f(text[i])` mà `f` chỉ phụ thuộc ký tự ⇒ **gom bằng `Counter`**.
- Bài interactive ⇒ đếm số lần gọi API là thước đo chi phí; tự viết lớp giả lập để test.
- Thuộc **một** mẫu nhị phân và dùng mãi (khuyên: `while lo <= hi` + biến `ans`), tránh trộn mẫu gây lặp vô hạn.
- Luôn tự hỏi: nếu **không có** đáp án thì trả gì (`−1`)?

## 15. Biến thể của bài

1. **Nhiều dòng:** `text` gồm nhiều dòng cố định; mọi dòng phải `≤ w` và tổng chiều cao `số_dòng · getHeight ≤ h` — vẫn đơn điệu (Bài tập 5).
2. **Tự xuống dòng (word wrap):** chữ được ngắt dòng theo từ; với cỡ cố định, đếm số dòng cần bằng tham lam, rồi nhị phân trên cỡ (giống **1891 Cutting Ribbons**, **875 Koko Eating Bananas**).
3. **`fonts` là đoạn liên tục `[1, 10⁵]`**: nhị phân trên giá trị thay vì chỉ số.
4. **Không có tính đơn điệu** (FontInfo tùy ý): bắt buộc thử mọi cỡ — chỉ còn tối ưu bằng Counter (Cách 2).
5. Bài liên quan: **374 Guess Number Higher or Lower**, **278 First Bad Version** (interactive + nhị phân), **1011 Capacity To Ship Packages Within D Days**, **410 Split Array Largest Sum** (binary search on answer).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time (số lần gọi API) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cỡ | Duyệt hết `fonts`, cộng trực tiếp | `O(F·T)` | `O(1)` | Kiểm chứng, input nhỏ |
| 2. Quét + Counter | Từ lớn xuống, ≤ 26 lần gọi mỗi cỡ | `O(T + Σ·F)` | `O(Σ)` | Khi điều kiện **không** đơn điệu |
| 3. Nhị phân | "True cuối cùng" trên chỉ số | `O(T log F)` | `O(1)` | Đủ qua; dễ viết |
| 4. Nhị phân + Counter | Cả hai insight | `O(T + Σ log F)` | `O(Σ)` | **Bản nộp khuyên dùng** |
| 5. `bisect` + `key` | Như 4, viết gọn | `O(T + Σ log F)` | `O(Σ)` | Python ≥ 3.10, muốn code ngắn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

Mọi bài dùng `FontInfo` giả lập ở trên (`getWidth(fs, ch) = fs + [ch ≥ 'n']`, `getHeight(fs) = fs`).

**Bài 1 (đoán output):** (a) `text="aaa", w=30, h=10, fonts=[5,10,15]`; (b) `text="zzz", w=30, h=10, fonts=[5,10,15]`; (c) `text="abc", w=2, h=100, fonts=[1,2]`; (d) `text="n", w=100, h=7, fonts=[1,3,5,7,9]`.

**Bài 2 (tính tay):** Với `text = "zebra"`, tính chiều rộng ở cỡ `10` và cỡ `12`. Với `w = 55`, `h = 20`, `fonts = [8, 10, 12]`, đáp án là bao nhiêu? Trace nhị phân (lo, hi, mid) bằng tay.

**Bài 3 (điền TODO):** Hoàn thiện `text_width(cnt, fs, fontInfo)` tính chiều rộng từ `Counter` với dừng sớm khi vượt `limit`.

**Bài 4 (tự viết lại):** Viết `last_true_generic(lo, hi, pred)` trả về số nguyên `x ∈ [lo, hi]` lớn nhất với `pred(x)` đúng (giả sử `pred` đơn điệu giảm), hoặc `lo − 1` nếu không có. Dùng nó để giải bài gốc.

**Bài 5 (mini challenge):** `max_font_lines(lines, w, h, fonts, fontInfo)`: hiển thị **nhiều dòng** (`lines` là list chuỗi), cùng một cỡ; mỗi dòng phải rộng `≤ w` và tổng chiều cao `len(lines) · getHeight(fs) ≤ h`. Trả cỡ lớn nhất hoặc `−1`, với số lần gọi `getWidth` là `O(Σ · log F)`.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def text_width(cnt, fs, fontInfo, limit):
    total = 0
    for c, k in cnt.items():
        # TODO: cộng getWidth(fs, c) * k; nếu total > limit thì trả về ngay
        pass
    return total

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def last_true_generic(lo, hi, pred):
    # TODO: mẫu while lo <= hi với biến ans = lo - 1
    pass

def max_font_lines(lines, w, h, fonts, fontInfo):
    # TODO: một Counter cho mỗi dòng; fits(fs): chiều cao tổng + mọi dòng đều <= w
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính chiều rộng `= len·fs + số ký tự ≥ 'n'`. Mức 2: kiểm tra chiều cao `fs ≤ h`. Mức 3: (a) `10` (rộng 30 khít; cỡ 15 cao quá); (b) `5` (cỡ 10 rộng 33 > 30); (c) `-1` (cỡ 1 rộng 3 > 2); (d) `7` (rộng `fs + 1 ≤ 100`, cao `≤ 7`).

**Bài 2.** Mức 1: `'z'`, `'r'` thuộc `n..z` ⇒ +2. Mức 2: cỡ 10 rộng `52`, cỡ 12 rộng `62`. Mức 3: `lo=0, hi=2, mid=1` (cỡ 10: vừa) ⇒ `lo=2`; `mid=2` (cỡ 12: 62 > 55) ⇒ `hi=1`; dừng, đáp án `10`.

**Bài 3.** Mức 1: `total += fontInfo.getWidth(fs, c) * k`. Mức 2: `if total > limit: return total`. Mức 3: hàm gọi chỉ cần so `text_width(...) <= w`.

**Bài 4.** Mức 1: `ans = lo − 1` ban đầu. Mức 2: `pred(mid)` đúng ⇒ `ans = mid; lo = mid + 1`. Mức 3: bài gốc: `i = last_true_generic(0, F − 1, lambda i: fits(fonts[i]))`, trả `fonts[i]` nếu `i ≥ 0`.

**Bài 5.** Mức 1: tính trước `cnts = [Counter(line) for line in lines]`. Mức 2: `fits(fs)`: `len(lines) * getHeight(fs) ≤ h` và với mỗi dòng `text_width ≤ w`. Mức 3: gom **trùng ký tự giữa các dòng** không được (mỗi dòng có giới hạn riêng), nhưng có thể gom các dòng giống hệt nhau; số lần gọi tổng là `O(Σ_line Σ · log F)`. Kiểm chứng bằng brute force thử mọi cỡ.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
b1 = [Solution().maxFont(t, w, h, f, FontInfo()) for t, w, h, f in
      [("aaa", 30, 10, [5, 10, 15]), ("zzz", 30, 10, [5, 10, 15]), ("abc", 2, 100, [1, 2]), ("n", 100, 7, [1, 3, 5, 7, 9])]]
print("Bài 1:", b1); assert b1 == [10, 5, -1, 7]
fi = FontInfo()
w10 = sum(fi.getWidth(10, c) for c in "zebra"); w12 = sum(fi.getWidth(12, c) for c in "zebra")
print("Bài 2: rộng cỡ 10 =", w10, ", cỡ 12 =", w12)
assert (w10, w12) == (52, 62)
assert trace_binary("zebra", 55, 20, [8, 10, 12], FontInfo()) == 10

Bài 1: [10, 5, -1, 7]
Bài 2: rộng cỡ 10 = 52 , cỡ 12 = 62
cnt = {'z': 1, 'e': 1, 'b': 1, 'r': 1, 'a': 1}
vòng 1: mid=1, cỡ= 10, cao= 10  , rộng=52, vừa=True  -> lo=2, hi=2, ans=10
vòng 2: mid=2, cỡ= 12, cao= 12  , rộng=62, vừa=False -> lo=2, hi=1, ans=10
Kết quả: 10 | getWidth gọi 10 lần


In [29]:
# Bài 3, 4 — đáp án
def text_width(cnt, fs, fontInfo, limit):
    total = 0
    for c, k in cnt.items():
        total += fontInfo.getWidth(fs, c) * k
        if total > limit:
            return total
    return total

def last_true_generic(lo, hi, pred):
    ans = lo - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if pred(mid):
            ans, lo = mid, mid + 1
        else:
            hi = mid - 1
    return ans

def max_font_via_generic(text, w, h, fonts, fontInfo):
    cnt = Counter(text)
    def fits(fs):
        return fontInfo.getHeight(fs) <= h and text_width(cnt, fs, fontInfo, w) <= w
    i = last_true_generic(0, len(fonts) - 1, lambda i: fits(fonts[i]))
    return fonts[i] if i >= 0 else -1

assert last_true_generic(1, 100, lambda x: x * x <= 50) == 7
assert last_true_generic(5, 9, lambda x: False) == 4
rng = random.Random(77)
for _ in range(400):
    s = rng.randrange(10 ** 9)
    text = "".join(rng.choice("abcxyz") for _ in range(rng.randint(1, 20)))
    fonts = sorted(rng.sample(range(1, 150), rng.randint(1, 20)))
    w, h = rng.randint(1, 2000), rng.randint(1, 200)
    assert max_font_via_generic(text, w, h, fonts, RandomFontInfo(random.Random(s))) == \
           max_font_bruteforce(text, w, h, fonts, RandomFontInfo(random.Random(s)))
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [30]:
# Bài 5 — đáp án
def max_font_lines(lines, w, h, fonts, fontInfo):
    cnts = [Counter(line) for line in lines]
    L = len(lines)
    def fits(fs):
        if L * fontInfo.getHeight(fs) > h:
            return False
        return all(text_width(c, fs, fontInfo, w) <= w for c in cnts)
    lo, hi, ans = 0, len(fonts) - 1, -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if fits(fonts[mid]):
            ans, lo = fonts[mid], mid + 1
        else:
            hi = mid - 1
    return ans

def lines_brute(lines, w, h, fonts, fontInfo):
    ans = -1
    for fs in fonts:
        if len(lines) * fontInfo.getHeight(fs) <= h and all(sum(fontInfo.getWidth(fs, c) for c in ln) <= w for ln in lines):
            ans = fs
    return ans

print("Bài 5 ví dụ:", max_font_lines(["hello", "world"], 80, 20, [6, 8, 10, 12], FontInfo()))
rng = random.Random(55)
for _ in range(300):
    s = rng.randrange(10 ** 9)
    lines = ["".join(rng.choice("abmnz") for _ in range(rng.randint(1, 10))) for _ in range(rng.randint(1, 4))]
    fonts = sorted(rng.sample(range(1, 120), rng.randint(1, 15)))
    w, h = rng.randint(1, 800), rng.randint(1, 400)
    assert max_font_lines(lines, w, h, fonts, RandomFontInfo(random.Random(s))) == lines_brute(lines, w, h, fonts, RandomFontInfo(random.Random(s)))
print("Bài 5: OK")

Bài 5 ví dụ: 10
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "binary search on answer — True cuối cùng":**

```python
cnt = Counter(text)
def fits(fs):
    if fontInfo.getHeight(fs) > h: return False
    return sum(fontInfo.getWidth(fs, c) * k for c, k in cnt.items()) <= w

lo, hi, ans = 0, len(fonts) - 1, -1
while lo <= hi:
    mid = (lo + hi) // 2
    if fits(fonts[mid]): ans = fonts[mid]; lo = mid + 1
    else:                hi = mid - 1
return ans
```

**Checklist:**
- [ ] Kiểm tra **cả** chiều cao `≤ h` **và** chiều rộng `≤ w` (dùng `<=`, chấp nhận vừa khít)?
- [ ] Trả về **cỡ chữ** `fonts[i]`, không phải chỉ số `i`?
- [ ] Không cỡ nào vừa ⇒ trả `-1`?
- [ ] Mẫu nhị phân nhất quán (không `lo = mid` đi với `mid = (lo+hi)//2`)?
- [ ] Gom ký tự bằng `Counter` để giảm số lần gọi `getWidth`?
- [ ] Không tự định nghĩa lại lớp `FontInfo` trong bài nộp?
- [ ] Đã test: 1 cỡ, vừa khít, chỉ chiều cao chặn, không cỡ nào vừa, mọi cỡ đều vừa, FontInfo ngẫu nhiên đơn điệu?